# Faizaan Ali | HW5 | Task 3

## Part 1: Dataset Setup

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset
from nltk.translate.bleu_score import corpus_bleu
from collections import Counter
import math
import re

# Load a SMALL subset to prevent GPU OOM
dataset = load_dataset("wmt14", "fr-en", split="train[:4000]")

pairs = [
    (
        x["translation"]["en"].lower(),
        x["translation"]["fr"].lower()
    )
    for x in dataset
]

df = pd.DataFrame(pairs, columns=["en", "fr"])

print("Dataset size:", len(df))
df.head()

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

fr-en/train-00000-of-00030.parquet:   0%|          | 0.00/252M [00:00<?, ?B/s]

fr-en/train-00001-of-00030.parquet:   0%|          | 0.00/241M [00:00<?, ?B/s]

fr-en/train-00002-of-00030.parquet:   0%|          | 0.00/243M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/40836715 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3003 [00:00<?, ? examples/s]

Dataset size: 4000


,en,fr
0,resumption of the session,reprise de la session
1,i declare resumed the session of the european ...,je déclare reprise la session du parlement eur...
2,"although, as you will have seen, the dreaded '...","comme vous avez pu le constater, le grand ""bog..."
3,you have requested a debate on this subject in...,vous avez souhaité un débat à ce sujet dans le...
4,"in the meantime, i should like to observe a mi...","en attendant, je souhaiterais, comme un certai..."


In [ ]:
def tokenize(s):
    return re.findall(r"\b\w+\b", s.lower())

df["en_tok"] = df.en.apply(tokenize)
df["fr_tok"] = df.fr.apply(tokenize)

def build_vocab(sentences):
    counter = Counter([w for s in sentences for w in s])
    vocab = ["<pad>", "<sos>", "<eos>", "<unk>"]
    vocab += list(counter.keys())
    stoi = {w:i for i,w in enumerate(vocab)}
    itos = {i:w for w,i in stoi.items()}
    return stoi, itos

en_stoi,en_itos = build_vocab(df.en_tok)
fr_stoi,fr_itos = build_vocab(df.fr_tok)

def encode(sent, vocab):
    return [vocab.get(w,vocab["<unk>"]) for w in sent]

df["en_ids"] = df.en_tok.apply(lambda s: encode(s,en_stoi))
df["fr_ids"] = df.fr_tok.apply(lambda s: encode(s,fr_stoi))

In [ ]:
MAX_LEN = 15

def pad(seq):
    temp_seq = [en_stoi["<sos>"]] + seq + [en_stoi["<eos>"]]
    padded_seq = temp_seq[:MAX_LEN] + [0] * max(0, MAX_LEN - len(temp_seq))
    return padded_seq

def pad_fr(seq):
    temp_seq = [fr_stoi["<sos>"]] + seq + [fr_stoi["<eos>"]]
    padded_seq = temp_seq[:MAX_LEN] + [0] * max(0, MAX_LEN - len(temp_seq))
    return padded_seq

X = np.array(df["en_ids"].apply(pad).tolist())
Y = np.array(df["fr_ids"].apply(pad_fr).tolist())

train_size = int(0.9 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
Y_train, Y_test = Y[:train_size], Y[train_size:]

This preprocessing pipeline prepares the English–French dataset for training a sequence-to-sequence model. First, sentences are tokenized into lowercase word tokens using a simple regex-based tokenizer. Then, separate vocabularies are built for English and French, mapping each word to a unique index while including special tokens (pad, sos, eos, unk).

Next, each sentence is encoded into a sequence of integers using these vocabularies. To make sure that there is uniform input size, sequences are padded and truncated to a fixed maximum length (MAX_LEN = 15), while also adding start (sos) and end (eos) tokens.

Finally, the processed data is converted into NumPy arrays and split into training (90%) and test (10%) sets, producing input-output pairs for our machine translation model training.

## Scaled Dot Product Attention

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    dk = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(dk, dtype=Q.dtype, device=Q.device))

    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)  # mask out future tokens

    weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    return output, weights

# demo
Q = torch.randn(4,8)
K = torch.randn(4,8)
V = torch.randn(4,8)

# Ensure demo works with torch tensors
scaled_dot_product_attention(Q,K,V)

(tensor([[ 0.2507, -0.7006,  0.7957, -0.4675, -0.6098, -0.2623,  0.6450, -0.4675],
         [ 0.5980, -0.3622,  0.3879,  0.6655, -0.5253, -0.2497,  0.6636, -0.8520],
         [ 0.4685, -0.0044,  0.1550,  0.7120,  0.0707,  0.2433,  0.1048, -0.9546],
         [ 0.5289, -0.1570,  0.2342,  0.6662, -0.1725,  0.0501,  0.3346, -0.8822]]),
 tensor([[0.2957, 0.1062, 0.4798, 0.1183],
         [0.1028, 0.3898, 0.1683, 0.3391],
         [0.0566, 0.2010, 0.0788, 0.6636],
         [0.0901, 0.2668, 0.1111, 0.5320]]))

The `scaled_dot_product_attention` function computes how much each element in a sequence should “pay attention” to others. It first takes the dot product between queries `Q` and keys `K` (via `Q @ Kᵀ`) to measure similarity between tokens, then scales these scores by √dₖ (where dₖ is the key dimension) to prevent values from becoming too large and destabilizing training. If a `mask` is provided (in the decoder), certain positions are set to a very negative value so they don’t influence the result (this enforces causality by hiding future tokens). The scores are then passed through a softmax to produce attention weights (probabilities that sum to 1), which determine how important each token is relative to others. Finally, these weights are used to compute a weighted sum of the values `V`, producing the output representation. In short, it lets each token dynamically aggregate information from other relevant tokens in the sequence.


In [ ]:
class Encoder(nn.Module):
    def __init__(self,vocab,emb=64,hidden=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab,emb)
        self.rnn = nn.GRU(emb,hidden,batch_first=True)

    def forward(self,x):
        e = self.embedding(x)
        out,h = self.rnn(e)
        return out,h


class Attention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, query, keys, values):
        context, weights = scaled_dot_product_attention(query, keys, values)
        return context, weights


class Decoder(nn.Module):
    def __init__(self, vocab, emb=64, hidden=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab, emb)
        self.rnn = nn.GRU(emb + hidden, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab)
        self.attn = Attention()

    def forward(self, x, h, enc_out):
        e = self.embedding(x)
        context, _ = self.attn(e, enc_out, enc_out)
        inp = torch.cat((e, context), dim=2)
        out, h = self.rnn(inp, h)
        out = self.fc(out)
        return out, h


class Seq2Seq(nn.Module):
    def __init__(self,enc,dec):
        super().__init__()
        self.enc = enc
        self.dec = dec

    def forward(self,src,trg):
        enc_out,h = self.enc(src)
        out,_ = self.dec(trg,h,enc_out)
        return out

This model is a sequence-to-sequence (seq2seq) architecture with attention for machine translation. The encoder first embeds the input sentence and processes it with a GRU to produce hidden states (`enc_out`) and a final context vector (`h`). The decoder then takes the target sequence, embeds it, and uses our defined `scaled_dot_product_attention` to compare the decoder inputs (queries) with the encoder outputs (keys/values), creating a context vector that highlights relevant parts of the input sentence. This context is then concatenated with the decoder embeddings and passed through another GRU, followed by a linear layer to generate output word probabilities. Overall, the attention mechanism allows the decoder to dynamically focus on different parts of the input at each step instead of relying only on a single fixed context vector.


In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

device = "cuda" if torch.cuda.is_available() else "cpu"

# Instantiate Encoder and Decoder with matching embedding and hidden dimensions
# Set emb to 128 to match the default hidden size
enc = Encoder(len(en_stoi), emb=128, hidden=128).to(device)
dec = Decoder(len(fr_stoi), emb=128, hidden=128).to(device)
model = Seq2Seq(enc,dec).to(device)

opt = optim.Adam(model.parameters(),lr=1e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

Xtr = torch.tensor(X_train).to(device)
Ytr = torch.tensor(Y_train).to(device)

for epoch in range(100):
    model.train()
    opt.zero_grad()

    out = model(Xtr,Ytr[:,:-1])

    loss = loss_fn(
        out.reshape(-1,out.shape[-1]),
        Ytr[:,1:].reshape(-1)
    )

    loss.backward()
    opt.step()

    print("epoch",epoch,"loss",loss.item())

model.eval()

Xte = torch.tensor(X_test).to(device)
Yte = torch.tensor(Y_test).to(device)

with torch.no_grad():
    pred = model(Xte,Yte[:,:-1]).argmax(-1).cpu().numpy()

smooth = SmoothingFunction().method1

def decode(ids):
    return [fr_itos.get(i, "") for i in ids if i not in [0, fr_stoi["<sos>"], fr_stoi["<eos>"]]]

refs = [[decode(y)] for y in Y_test]
hyps = [decode(y) for y in pred]

bleu = corpus_bleu(refs, hyps, smoothing_function=smooth)

print("BLEU:", bleu)

epoch 0 loss 9.099485397338867
epoch 1 loss 9.05963134765625
epoch 2 loss 9.019384384155273
epoch 3 loss 8.975956916809082
epoch 4 loss 8.926589965820312
epoch 5 loss 8.86832046508789
epoch 6 loss 8.7977876663208
epoch 7 loss 8.71116828918457
epoch 8 loss 8.604327201843262
epoch 9 loss 8.473384857177734
epoch 10 loss 8.316051483154297
epoch 11 loss 8.133781433105469
epoch 12 loss 7.933911323547363
epoch 13 loss 7.729526042938232
epoch 14 loss 7.535094261169434
epoch 15 loss 7.360456466674805
epoch 16 loss 7.208405017852783
epoch 17 loss 7.076874732971191
epoch 18 loss 6.962348937988281
epoch 19 loss 6.861910343170166
epoch 20 loss 6.773784637451172
epoch 21 loss 6.6970086097717285
epoch 22 loss 6.630858898162842
epoch 23 loss 6.57444429397583
epoch 24 loss 6.526607513427734
epoch 25 loss 6.485998153686523
epoch 26 loss 6.451224327087402
epoch 27 loss 6.421111106872559
epoch 28 loss 6.394914627075195
epoch 29 loss 6.372167110443115
epoch 30 loss 6.352395534515381
epoch 31 loss 6.3349876

In [ ]:
# Show sample translations

def decode(ids, vocab):
    words = []
    for i in ids:
        if i == 0:
            continue
        w = vocab.get(i, "")
        if w in ["<sos>", "<eos>"]:
            continue
        words.append(w)
    return words

num_samples = 5

for i in range(num_samples):

    src_sentence = decode(X_test[i], en_itos)
    true_translation = decode(Y_test[i], fr_itos)
    pred_translation = decode(pred[i], fr_itos)

    print("EN :", " ".join(src_sentence))
    print("FR (true):", " ".join(true_translation))
    print("FR (pred):", " ".join(pred_translation))
    print("-"*50)

EN : i think that the report drawn up by the wise men has been useful
FR (true): je crois que le rapport des sages a été utile notamment pour le parlement
FR (pred): je voudrais que la président de le de de de de de l président
--------------------------------------------------
EN : we have submitted an amendment to this effect
FR (true): nous avons déposé un amendement à ce sujet
FR (pred): je avons de de de la que de de de de de de
--------------------------------------------------
EN : from the many points made by mr van hulten in his report i would
FR (true): je voudrais insister sur quelques uns des nombreux points du rapport de m van
FR (pred): je voudrais de de la de de le de de que de la que
--------------------------------------------------
EN : firstly i think that the commission should pay much more attention to the proper
FR (true): je pense tout d abord que la commission devrait accorder beaucoup plus d attention
FR (pred): je voudrais que de une de la commission de de de 

The training loss steadily decreases from about 9.09 to 5.47 over 100 epochs, indicating that the model is learning and improving its fit to the training data. However, despite this reduction in loss, the BLEU score remains very low (~0.01), showing that the model performs poorly on actual translation quality. The sample outputs confirm this: predictions are largely repetitive, dominated by common tokens (e.g., “je”, “de”, “la”), and fail to capture the meaning of the input sentences. This suggests that while the model is optimizing the objective function, it is underfitting or learning shallow patterns, likely due to limited model capacity and a very insufficient amount training data (only 4000 item subset).


## Custom Transformer

In [ ]:
import torch
import torch.nn as nn
import math

# Positional Encoding (Sinusoidal)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)

        for pos in range(max_len):
            for i in range(0, d_model, 2):
                pe[pos, i] = math.sin(pos / (10000 ** (i / d_model)))
                if i + 1 < d_model:
                    pe[pos, i + 1] = math.cos(pos / (10000 ** (i / d_model)))

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=64, heads=2):
        super().__init__()
        self.heads = heads
        self.d_k = d_model // heads

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.fc = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V, mask=None):
        B = Q.size(0)

        Q = self.Wq(Q).view(B, -1, self.heads, self.d_k).transpose(1, 2)  # [B, heads, seq_len, d_k]
        K = self.Wk(K).view(B, -1, self.heads, self.d_k).transpose(1, 2)
        V = self.Wv(V).view(B, -1, self.heads, self.d_k).transpose(1, 2)

        if mask is not None:
            mask = mask.expand(-1, self.heads, Q.size(2), mask.size(-1))

        out, _ = scaled_dot_product_attention(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(B, -1, self.heads * self.d_k)
        return self.fc(out)


# Encoder Layer
class EncoderLayer(nn.Module):
    def __init__(self, d_model=64, heads=2, ff=128):
        super().__init__()

        self.attn = MultiHeadAttention(d_model, heads)
        self.norm1 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, ff),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        x = self.norm1(x + self.attn(x, x, x, mask))
        x = self.norm2(x + self.ff(x))
        return x


# Decoder Layer
class DecoderLayer(nn.Module):
    def __init__(self, d_model=64, heads=2, ff=128):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, heads)
        self.norm1 = nn.LayerNorm(d_model)

        self.cross_attn = MultiHeadAttention(d_model, heads)
        self.norm2 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, ff),
            nn.ReLU(),
            nn.Linear(ff, d_model)
        )
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, enc_output, src_mask=None, trg_mask=None):
        x = self.norm1(x + self.self_attn(x, x, x, trg_mask))
        x = self.norm2(x + self.cross_attn(x, enc_output, enc_output, src_mask))
        x = self.norm3(x + self.ff(x))
        return x


# Full Transformer
class SimpleTransformer(nn.Module):
    def __init__(self, src_vocab, trg_vocab, d_model=64, heads=2, ff=128):
        super().__init__()

        self.emb_src = nn.Embedding(src_vocab, d_model)
        self.emb_trg = nn.Embedding(trg_vocab, d_model)

        self.pos = PositionalEncoding(d_model)

        # 2 Encoder layers
        self.enc1 = EncoderLayer(d_model, heads, ff)
        self.enc2 = EncoderLayer(d_model, heads, ff)

        # 2 Decoder layers
        self.dec1 = DecoderLayer(d_model, heads, ff)
        self.dec2 = DecoderLayer(d_model, heads, ff)

        self.fc_out = nn.Linear(d_model, trg_vocab)

    def make_trg_mask(self, trg):
        B, T = trg.shape

        # padding mask
        pad_mask = (trg != 0).unsqueeze(1).unsqueeze(2)  # [B,1,1,T]

        # causal mask
        causal_mask = torch.tril(torch.ones((T, T), device=trg.device)).bool()
        causal_mask = causal_mask.unsqueeze(0).unsqueeze(1)  # [1,1,T,T]

        # combine
        return pad_mask & causal_mask

    def forward(self, src, trg, src_mask=None):
        # Create src_mask if not provided
        if src_mask is None:
            src_mask = (src != 0).unsqueeze(1).unsqueeze(2)  # [B,1,1,seq_len]

        trg_mask = self.make_trg_mask(trg)

        # Encoder
        enc = self.pos(self.emb_src(src) * math.sqrt(self.emb_src.embedding_dim))
        enc = self.enc1(enc, src_mask)
        enc = self.enc2(enc, src_mask)

        # Decoder
        dec = self.pos(self.emb_trg(trg) * math.sqrt(self.emb_trg.embedding_dim))
        dec = self.dec1(dec, enc, src_mask, trg_mask)
        dec = self.dec2(dec, enc, src_mask, trg_mask)

        out = self.fc_out(dec)
        return out

In [ ]:
import torch.optim as optim
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

device = "cuda" if torch.cuda.is_available() else "cpu"

# Instantiate Transformer
model_t = SimpleTransformer(len(en_stoi), len(fr_stoi), d_model=64, heads=2, ff=128).to(device)

opt = optim.Adam(model_t.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)

Xtr_t = torch.tensor(X_train).to(device)
Ytr_t = torch.tensor(Y_train).to(device)
Xte_t = torch.tensor(X_test).to(device)
Yte_t = torch.tensor(Y_test).to(device)

batch_size = 64

for epoch in range(100):
    model_t.train()

    for i in range(0, len(Xtr_t), batch_size):
        src_batch = Xtr_t[i:i+batch_size]
        trg_batch = Ytr_t[i:i+batch_size]

        opt.zero_grad()

        # forward pass (src_mask is created internally)
        out = model_t(src_batch, trg_batch[:, :-1])

        loss = loss_fn(
            out.reshape(-1, out.shape[-1]),
            trg_batch[:, 1:].reshape(-1)
        )

        loss.backward()
        opt.step()

    print(f"Epoch {epoch} Loss: {loss.item():.4f}")

# Sequential evaluation
def generate(model, src, max_len=20, start_token=fr_stoi["<sos>"]):
    model.eval()
    src = torch.tensor(src).unsqueeze(0).to(device)  # add batch dim
    generated = [start_token]

    # Create src_mask for this source
    src_mask = (src != 0).unsqueeze(1).unsqueeze(2)

    for _ in range(max_len):
        trg_tensor = torch.tensor(generated).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model(src, trg_tensor, src_mask=src_mask)  # pass src_mask
            probs = torch.softmax(out[:, -1, :] / 0.8, dim=-1)
            next_token = torch.multinomial(probs, 1).item()
            generated.append(next_token)
            if next_token == fr_stoi["<eos>"]:
                break
    return generated

# Evaluation using sequential generation
model_t.eval()
pred_sequences = []

for i in range(len(X_test[:500])):  # limit to 500 for BLEU
    src_seq = X_test[i]
    pred_ids = generate(model_t, src_seq, max_len=50)
    pred_sequences.append(pred_ids)

# Decode for BLEU
refs = [[decode(y, fr_itos)] for y in Y_test[:500]]
hyps = [decode(y, fr_itos) for y in pred_sequences]

smooth = SmoothingFunction().method1
bleu = corpus_bleu(refs, hyps, smoothing_function=smooth)
print("Transformer BLEU (sequential generation):", bleu)

# Sample translations
num_samples = 5
for i in range(num_samples):
    src_sentence = decode(X_test[i], en_itos)
    true_translation = decode(Y_test[i], fr_itos)
    pred_translation = decode(pred_sequences[i], fr_itos)

    print("EN :", " ".join(src_sentence))
    print("FR (true):", " ".join(true_translation))
    print("FR (pred):", " ".join(pred_translation))
    print("-"*50)

Epoch 0 Loss: 6.7971
Epoch 1 Loss: 6.2937
Epoch 2 Loss: 5.7368
Epoch 3 Loss: 5.2738
Epoch 4 Loss: 4.9208
Epoch 5 Loss: 4.5775
Epoch 6 Loss: 4.2945
Epoch 7 Loss: 4.0232
Epoch 8 Loss: 3.7333
Epoch 9 Loss: 3.4955
Epoch 10 Loss: 3.2561
Epoch 11 Loss: 3.0391
Epoch 12 Loss: 2.8439
Epoch 13 Loss: 2.6511
Epoch 14 Loss: 2.4968
Epoch 15 Loss: 2.3086
Epoch 16 Loss: 2.2096
Epoch 17 Loss: 2.1144
Epoch 18 Loss: 2.0078
Epoch 19 Loss: 1.9675
Epoch 20 Loss: 1.8913
Epoch 21 Loss: 1.8628
Epoch 22 Loss: 1.7846
Epoch 23 Loss: 1.7874
Epoch 24 Loss: 1.7151
Epoch 25 Loss: 1.7083
Epoch 26 Loss: 1.7229
Epoch 27 Loss: 1.6978
Epoch 28 Loss: 1.6782
Epoch 29 Loss: 1.6616
Epoch 30 Loss: 1.6619
Epoch 31 Loss: 1.6624
Epoch 32 Loss: 1.6524
Epoch 33 Loss: 1.6372
Epoch 34 Loss: 1.6381
Epoch 35 Loss: 1.6283
Epoch 36 Loss: 1.6106
Epoch 37 Loss: 1.6034
Epoch 38 Loss: 1.6444
Epoch 39 Loss: 1.6094
Epoch 40 Loss: 1.5994
Epoch 41 Loss: 1.5986
Epoch 42 Loss: 1.5981
Epoch 43 Loss: 1.6040
Epoch 44 Loss: 1.6014
Epoch 45 Loss: 1.583

This last section of code implements a simplified Transformer-based sequence-to-sequence model for English-to-French translation using PyTorch. It includes key components of the Transformer architecture such as sinusoidal positional encoding, multi-head self-attention built on a scaled dot-product attention function, encoder and decoder layers with residual connections and layer normalization, and [masking](https://medium.com/analytics-vidhya/masking-in-transformers-self-attention-mechanism-bad3c9ec235c) to enforce both padding and causal constraints during decoding. The model is trained using teacher forcing with cross-entropy loss (including label smoothing) and optimized with Adam.

As for the results, the loss steadily decreases from about 6.8 to around 1.47, indicating that the model is again successfully learning token-level patterns and improving its predictions over time, although it plateaus in later epochs, suggesting limited model capacity or dataset size. When evaluated using sequential generation and BLEU score, the model performs poorly (BLEU ≈ 0.0116), revealing a gap between training performance and actual translation quality. The generated outputs demonstrate partial understanding of sentence structure and vocabulary (producing readable openings like “je pense…”), but they often become ungrammatical, repetitive, or semantically incorrect as the sequence progresses. This discrepancy highlights common issues in sequence generation tasks such as exposure bias, insufficient model size, and imperfect decoding strategies, where the model can minimize training loss yet still struggle to produce coherent and accurate full-sentence translations.

## Comparison

The Seq2Seq model and the Transformer exhibit different failure modes in their translations. The Seq2Seq outputs are highly repetitive and degenerate, usually looking like short phrases dominated by frequent tokens like “de” and “la". This shows that the model struggles with long-range dependencies and suffers from exposure bias, leading to early loss of meaningful structure. In contrast, the Transformer produces more diverse and fluent-looking sentences, demonstrating a better grasp of vocabulary and global sentence patterns (phrases like “je pense…” or “nous avons…”), probably due to its self-attention mechanism. However, its outputs are often overly long, semantically incorrect, and sometimes nonsensical, suggesting that while it captures broader context, it still lacks precise alignment and control during generation. Overall, the Seq2Seq model fails by being too simplistic and repetitive, whereas the Transformer fails in a more sophisticated way by producing grammatically richer but incoherent or inaccurate translations.
